In [1]:
!pip install emoji
!pip install nltk

In [2]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
import re
import string
import emoji
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
not_to_remove = {"not", "nor", "no", "or", "and"}
stop_words = stop_words - not_to_remove

stemmer = PorterStemmer()

def preprocess_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Emoji → text
    text = emoji.demojize(text)

    # Replace symbols
    text = re.sub(r'\$', ' dollar ', text)
    text = re.sub(r'\?', ' question ', text)
    text = re.sub(r'\&', ' and ', text)
    text = re.sub(r'\£', ' pound ', text)
    text = re.sub(r'\d+', ' num ', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove stopwords
    words = [w for w in word_tokenize(text) if w not in stop_words]

    # Stemming
    words = [stemmer.stem(w) for w in words]
    text = " ".join(words)

    return text

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
import pandas as pd
import numpy as np

In [5]:
train = pd.read_csv('SMS_train.csv',encoding='Latin1')
test = pd.read_csv('SMS_test.csv',encoding='Latin1')

In [6]:
train.drop(['S. No.'],axis=1,inplace=True)
test.drop(['S. No.'],axis=1,inplace=True)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000,ngram_range=(1,2))

In [8]:
train.columns

Index(['Message_body', 'Label'], dtype='object')

In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [10]:
x_train = tfidf.fit_transform(train['Message_body'])
y_train = le.fit_transform(train['Label'])
x_test = tfidf.transform(test['Message_body'])
y_test = le.transform(test['Label'])

In [11]:
!pip install imblearn

In [ ]:
from imblearn.over_sampling import SMOTE
x_train_resampled, y_train_resampled = SMOTE().fit_resample(x_train, y_train)

In [13]:
import joblib

joblib.dump(x_train, 'x_train.joblib')
joblib.dump(x_test, 'x_test.joblib')

joblib.dump(y_train, 'y_train.joblib')
joblib.dump(y_test, 'y_test.joblib')

joblib.dump(x_train_resampled, 'x_train_resampled.joblib')
joblib.dump(y_train_resampled, 'y_train_resampled.joblib')

joblib.dump(tfidf, 'tfidf.joblib')
joblib.dump(le, 'label_encoder.joblib')

['label_encoder.joblib']

In [14]:
text = "Congratulations! You've won a free ticket to the Bahamas! Click here to claim your prize: http://example.com/win"
preprocessed_text = preprocess_text(text)
x = tfidf.transform([preprocessed_text])
print(x)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4 stored elements and shape (1, 3000)>
  Coords	Values
  (0, 439)	0.4551089728779014
  (0, 1094)	0.3919409358768551
  (0, 2052)	0.4807834793082659
  (0, 2503)	0.6388312544124892
